## The MIDI Composer (Algorithmic Music Harmonization)

Melody $\rightarrow$ Chord Progression

In [1]:
import pickle
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

PAD_IDX = 0
SOS_IDX = 1
EOS_IDX = 2
REST_IDX = 3
NUM_SPECIAL_TOKENS = 4  # PAD, SOS, EOS, REST

def load_and_prep_jsb_pkl(filepath):
    with open(filepath, 'rb') as f:
        jsb_data = pickle.load(f, encoding='latin1')

    unique_chords = set()
    for split in ['train', 'valid', 'test']:
        for chorale in jsb_data[split]:
            for time_step in chorale:
                chord_tuple = tuple(sorted(time_step))
                unique_chords.add(chord_tuple)

    # FIX: Start chord indices at NUM_SPECIAL_TOKENS (=4), not 3.
    # Index 3 is REST_IDX, so starting at 3 caused a collision.
    chord2idx = {chord: idx + NUM_SPECIAL_TOKENS for idx, chord in enumerate(sorted(unique_chords))}
    idx2chord = {idx: chord for chord, idx in chord2idx.items()}

    return jsb_data, chord2idx, idx2chord

filepath = 'jsb-chorales-16th.pkl'
jsb_data, chord2idx, idx2chord = load_and_prep_jsb_pkl(filepath)

melody_vocab_size = 128 + NUM_SPECIAL_TOKENS  # 132
# FIX: Account for all 4 special tokens (was len(chord2idx) + 3).
harmony_vocab_size = len(chord2idx) + NUM_SPECIAL_TOKENS

print(f"Melody Vocab Size: {melody_vocab_size}")
print(f"Harmony Vocab Size: {harmony_vocab_size}")

Melody Vocab Size: 132
Harmony Vocab Size: 5349


In [2]:
class ChoraleDataset(Dataset):
    def __init__(self, data_split, chord2idx):
        self.data = data_split
        self.chord2idx = chord2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        chorale = self.data[idx]

        melody_tokens = [SOS_IDX]
        harmony_tokens = [SOS_IDX]

        for time_step in chorale:
            if len(time_step) == 0:
                melody_tokens.append(REST_IDX)
            else:
                melody_pitch = max(time_step)
                # Pitches 0–127 shifted by NUM_SPECIAL_TOKENS so they don't
                # collide with PAD/SOS/EOS/REST. Max token = 127+4 = 131 < 132.
                melody_tokens.append(melody_pitch + NUM_SPECIAL_TOKENS)

            chord_tuple = tuple(sorted(time_step))
            harmony_tokens.append(self.chord2idx[chord_tuple])

        melody_tokens.append(EOS_IDX)
        harmony_tokens.append(EOS_IDX)

        return torch.tensor(melody_tokens, dtype=torch.long), torch.tensor(harmony_tokens, dtype=torch.long)

train_dataset = ChoraleDataset(jsb_data['train'], chord2idx)
test_dataset = ChoraleDataset(jsb_data['test'], chord2idx)
val_dataset = ChoraleDataset(jsb_data['valid'], chord2idx)

In [3]:
def collate_batch(batch):
    melody_list, harmony_list = [], []
    for melody, harmony in batch:
        melody_list.append(melody)
        harmony_list.append(harmony)

    melody_padded = pad_sequence(melody_list, batch_first=True, padding_value=PAD_IDX)
    harmony_padded = pad_sequence(harmony_list, batch_first=True, padding_value=PAD_IDX)
    return melody_padded, harmony_padded

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  collate_fn=collate_batch)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

for melodies, harmonies in train_loader:
    print("Train Melody batch shape:", melodies.shape)
    print("Train Harmony batch shape:", harmonies.shape)
    break

Train Melody batch shape: torch.Size([32, 454])
Train Harmony batch shape: torch.Size([32, 454])


In [4]:
import torch
import torch.nn as nn
import math
import numpy as np
import pretty_midi

class MelodyHarmonizer(nn.Module):
    def __init__(
        self,
        melody_vocab_size,
        harmony_vocab_size,
        d_model=256,        
        nhead=8,
        num_layers=3,    
        dim_feedforward=1024,
        dropout=0.1,
        max_len=5000
    ):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len

        self.melody_embedding  = nn.Embedding(melody_vocab_size,  d_model, padding_idx=PAD_IDX)
        self.harmony_embedding = nn.Embedding(harmony_vocab_size, d_model, padding_idx=PAD_IDX)

        self.src_pos_embedding = nn.Embedding(max_len, d_model)
        self.tgt_pos_embedding = nn.Embedding(max_len, d_model)

        self.dropout = nn.Dropout(dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.output_projection = nn.Linear(d_model, harmony_vocab_size)

    def forward(
        self,
        src,
        tgt,
        tgt_mask=None,
        src_key_padding_mask=None,
        tgt_key_padding_mask=None,
        memory_key_padding_mask=None
    ):
        _, src_len = src.shape
        _, tgt_len = tgt.shape

        if src_len > self.max_len or tgt_len > self.max_len:
            raise ValueError(f"Sequence length exceeds max_len={self.max_len}")

        src_positions = torch.arange(src_len, device=src.device).unsqueeze(0)
        tgt_positions = torch.arange(tgt_len, device=tgt.device).unsqueeze(0)

        src_emb = self.melody_embedding(src)  * math.sqrt(self.d_model)
        tgt_emb = self.harmony_embedding(tgt) * math.sqrt(self.d_model)

        src_emb = src_emb + self.src_pos_embedding(src_positions)
        tgt_emb = tgt_emb + self.tgt_pos_embedding(tgt_positions)

        src_emb = self.dropout(src_emb)
        tgt_emb = self.dropout(tgt_emb)

        out = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask
        )
        return self.output_projection(out)

    def generate_square_subsequent_mask(self, sz):
        mask = torch.triu(torch.ones(sz, sz), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask

In [5]:
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MelodyHarmonizer(melody_vocab_size, harmony_vocab_size).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

Trainable parameters: 10,868,453


In [6]:


def train_model(model, dataloader, optimizer, criterion, epochs=20, device='cpu'):
    model.train()
    history = []
    for epoch in range(epochs):
        total_loss = 0.0
        n_batches = 0
        for melody, harmony in dataloader:
            melody  = melody.to(device=device, dtype=torch.long)
            harmony = harmony.to(device=device, dtype=torch.long)

            # Teacher forcing: decoder sees harmony[:, :-1], predicts harmony[:, 1:]
            tgt_input    = harmony[:, :-1]
            tgt_expected = harmony[:, 1:]

            tgt_mask = model.generate_square_subsequent_mask(tgt_input.size(1)).to(device)

            src_padding_mask = (melody == PAD_IDX)
            tgt_padding_mask = (tgt_input == PAD_IDX)

            optimizer.zero_grad()
            logits = model(
                melody,
                tgt_input,
                tgt_mask=tgt_mask,
                src_key_padding_mask=src_padding_mask,
                tgt_key_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask,
            )

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_expected.reshape(-1)
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            n_batches += 1

        avg_loss = total_loss / max(n_batches, 1)
        history.append(avg_loss)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_loss:.4f}")
    return history

In [7]:


def evaluate_model(model, dataloader, criterion, device='cpu'):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for melody, harmony in dataloader:
            melody  = melody.to(device=device, dtype=torch.long)
            harmony = harmony.to(device=device, dtype=torch.long)

            tgt_input    = harmony[:, :-1]
            tgt_expected = harmony[:, 1:]

            tgt_mask = model.generate_square_subsequent_mask(tgt_input.size(1)).to(device)
            src_padding_mask = (melody == PAD_IDX)
            tgt_padding_mask = (tgt_input == PAD_IDX)

            logits = model(
                melody,
                tgt_input,
                tgt_mask=tgt_mask,
                src_key_padding_mask=src_padding_mask,
                tgt_key_padding_mask=tgt_padding_mask,
                memory_key_padding_mask=src_padding_mask,
            )
            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                tgt_expected.reshape(-1)
            )
            total_loss += loss.item()
            n_batches += 1

    return total_loss / max(n_batches, 1)

In [8]:
def harmonize_melody(model, melody_sequence, sos_token=SOS_IDX, eos_token=EOS_IDX,
                     max_len=512, device='cpu'):
    """Greedy autoregressive generation. melody_sequence: 1-D tensor or list."""
    model.eval()
    model.to(device)

    if not isinstance(melody_sequence, torch.Tensor):
        melody_sequence = torch.tensor(melody_sequence)
    melody = melody_sequence.unsqueeze(0).to(device)  # (1, src_len)

    tgt = torch.tensor([[sos_token]], device=device)

    for _ in range(max_len):
        tgt_mask = model.generate_square_subsequent_mask(tgt.size(1)).to(device)
        with torch.no_grad():
            output = model(melody, tgt, tgt_mask=tgt_mask)
            next_token = torch.argmax(output[:, -1, :], dim=-1).item()
            tgt = torch.cat([tgt, torch.tensor([[next_token]], device=device)], dim=1)
            if next_token == eos_token:
                break

    return tgt.squeeze(0).tolist()

In [9]:

SPECIAL_TOKENS = {PAD_IDX, SOS_IDX, EOS_IDX, REST_IDX}

def tokens_to_midi(melody_tokens, harmony_tokens, idx2chord,
                   filename='bach_output.mid', tempo=100):
    """
    melody_tokens:  list of melody token IDs (with +4 offset; specials stripped here)
    harmony_tokens: list of chord token IDs (mapped via idx2chord)
    idx2chord:      dict mapping chord_token_id -> tuple of MIDI pitches
    """
    midi = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    melody_inst  = pretty_midi.Instrument(program=0, name='Soprano Melody')
    harmony_inst = pretty_midi.Instrument(program=0, name='Chord Accompaniment')

    seconds_per_beat = 60.0 / tempo
    step_duration = seconds_per_beat / 4.0  # 16th note

    # Melody — skip specials, undo the +4 offset
    for i, tok in enumerate(melody_tokens):
        if tok in SPECIAL_TOKENS:
            continue
        pitch = tok - NUM_SPECIAL_TOKENS
        if 0 <= pitch <= 127:
            start = i * step_duration
            end   = start + step_duration
            melody_inst.notes.append(
                pretty_midi.Note(velocity=100, pitch=pitch, start=start, end=end)
            )

    # Harmony — look up chord tuple, skip specials and unknowns
    for i, tok in enumerate(harmony_tokens):
        if tok in SPECIAL_TOKENS or tok not in idx2chord:
            continue
        chord_pitches = idx2chord[tok]
        if not chord_pitches:
            continue
        start = i * step_duration
        end   = start + step_duration
        for pitch in chord_pitches:
            if 0 <= pitch <= 127:
                harmony_inst.notes.append(
                    pretty_midi.Note(velocity=80, pitch=pitch, start=start, end=end)
                )

    midi.instruments.append(melody_inst)
    midi.instruments.append(harmony_inst)
    midi.write(filename)
    print(f"Saved generated audio to {filename}")

In [10]:


num_epochs = 20
best_val_loss = float('inf')

for epoch in range(num_epochs):
    train_history = train_model(
        model, train_loader, optimizer, criterion,
        epochs=1, device=device
    )
    val_loss = evaluate_model(model, val_loader, criterion, device=device)
    print(f"  -> Validation Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"  -> Saved new best model (val_loss={val_loss:.4f})")

print(f"\nTraining complete. Best validation loss: {best_val_loss:.4f}")

c:\MIX\ASU\SEM_4\LLM_Basics\phase2\Lib\site-packages\torch\nn\modules\activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch 1/1 | Train Loss: 8.4853


c:\MIX\ASU\SEM_4\LLM_Basics\phase2\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


  -> Validation Loss: 8.2954
  -> Saved new best model (val_loss=8.2954)
Epoch 1/1 | Train Loss: 8.1518
  -> Validation Loss: 8.0489
  -> Saved new best model (val_loss=8.0489)
Epoch 1/1 | Train Loss: 7.8914
  -> Validation Loss: 7.8153
  -> Saved new best model (val_loss=7.8153)
Epoch 1/1 | Train Loss: 7.6470
  -> Validation Loss: 7.5906
  -> Saved new best model (val_loss=7.5906)
Epoch 1/1 | Train Loss: 7.4159
  -> Validation Loss: 7.3753
  -> Saved new best model (val_loss=7.3753)
Epoch 1/1 | Train Loss: 7.2535
  -> Validation Loss: 7.1777
  -> Saved new best model (val_loss=7.1777)
Epoch 1/1 | Train Loss: 6.9982
  -> Validation Loss: 6.9854
  -> Saved new best model (val_loss=6.9854)
Epoch 1/1 | Train Loss: 6.8244
  -> Validation Loss: 6.8094
  -> Saved new best model (val_loss=6.8094)
Epoch 1/1 | Train Loss: 6.6776
  -> Validation Loss: 6.6509
  -> Saved new best model (val_loss=6.6509)
Epoch 1/1 | Train Loss: 6.4792
  -> Validation Loss: 6.5090
  -> Saved new best model (val_loss

In [11]:
# Example: harmonize the first melody from the validation set and save MIDI.

sample_melody, sample_harmony_gt = val_dataset[0]
predicted_harmony = harmonize_melody(
    model, sample_melody,
    sos_token=SOS_IDX, eos_token=EOS_IDX,
    max_len=sample_melody.size(0) + 16,
    device=device
)

print(f"Melody length: {sample_melody.size(0)}")
print(f"Predicted harmony length: {len(predicted_harmony)}")

tokens_to_midi(
    sample_melody.tolist(),
    predicted_harmony,
    idx2chord,
    filename='bach_output.mid',
    tempo=50
)

Melody length: 198
Predicted harmony length: 215
Saved generated audio to bach_output.mid


---
## The MIDI Composer (Algorithmic Music Harmonization) _ V2
- Soprano → Alto, 
- then Soprano + ground-truth Alto → Tenor, 
- then Soprano + ground-truth Alto + ground-truth Tenor → Bass